# Retrieval for a pricing agent: comparing chunking strategies

Stage 2 of the pricing agent. In stage 1 the agent knew only the numbers typed into its prompt. Here it gets a document to read: it retrieves passages from a price list and answers from those passages alone.

**The document:** [`pricing_guide.md`](pricing_guide.md), a fictional software price list with three products, 10 plans, bundles and billing policies. Its structure is what makes it a good test: plans in different products share names and prices, and every price appears only in a header.

**The question:** retrieval can only return what chunking produced, so how should the document be cut? This notebook compares the strategies below on the same questions, tests the agent on the most promising ones, and follows up each failure.

| Strategy | How it cuts |
| --- | --- |
| `fixed-300` | Fixed-size windows, cut at line breaks |
| `recursive-100/300/600` | Recursive splitting: paragraphs first, then lines, then words |
| `markdown-default` | One chunk per section, headers moved into metadata |
| `markdown-contextual` | The same sections, with product and plan names written back into the text |
| `semantic` | New chunk where embedding similarity between sentences drops |

## Setup

Install the dependencies from the repository root (`pip install -r requirements.txt`) and put your Groq API key in a `.env` file at the repository root (copy `.env.example`). `pricing_rag` loads the key when it is imported. The embedding model runs locally and needs no key.

In [1]:
from langchain_experimental.text_splitter import SemanticChunker

from pricing_rag import (
    MODEL, PRICING_PROMPT, SAMPLE_QUESTIONS, STRATEGIES, ask, build_keyword_index, build_rag_chain, build_store,
    compare_strategies, hypothetical_answer, keyword_search, load_guide, make_embeddings, make_llm,
    print_hits, split_recursive,
)

guide = load_guide()
embeddings = make_embeddings()
llm = make_llm()  # temperature 0, max_tokens 1024, low reasoning effort

print("Model:", MODEL)
print(f"Price list: {len(guide):,} characters, {guide.count(chr(10) + '### ')} sections")

Model: openai/gpt-oss-20b
Price list: 4,526 characters, 17 sections


## 1. What each strategy does to the document

The table counts what matters to a pricing agent: how big the chunks are, how many contain a price in their text, and whether any single chunk holds the whole Analytics Pro plan with its price.

In [2]:
chunks_by_strategy = {name: split(guide, embeddings) for name, split in STRATEGIES.items()}


def holds_whole_pro_plan(chunks):
    """True if one chunk has the Pro plan's price and its last feature."""
    return any("$99/month" in c.page_content and "Scheduled reports" in c.page_content for c in chunks)


print(f"{'Strategy':<22}{'Chunks':>7}{'Average':>9}{'Largest':>9}{'With a price':>14}{'Pro plan whole':>16}")
for name, chunks in chunks_by_strategy.items():
    lengths = [len(chunk.page_content) for chunk in chunks]
    with_price = sum("$" in chunk.page_content for chunk in chunks)
    print(f"{name:<22}{len(chunks):>7}{sum(lengths) / len(lengths):>9.0f}{max(lengths):>9}"
          f"{with_price:>14}{str(holds_whole_pro_plan(chunks)):>16}")

Strategy               Chunks  Average  Largest  With a price  Pro plan whole
fixed-300                  17      278      297            11           False
recursive-100              64       69       99            13           False
recursive-300              22      210      296            13           False
recursive-600              10      453      563             8            True
markdown-default           17      224      358             0           False
markdown-contextual        17      272      408            13            True
semantic                    8      564     2996             3            True


### Fixed windows ignore the plans

The second and third fixed chunks show the problem. The Pro plan starts in one chunk and finishes in the next, and the second half has no plan name above it, so its features could be credited to whichever plan comes next.

In [3]:
for chunk in chunks_by_strategy["fixed-300"][1:3]:
    print(f"--- {len(chunk.page_content)} chars ---")
    print(chunk.page_content)
    print()

--- 277 chars ---
- API access: Not included
- Custom branding: Not included
### Pro Plan - $99/month
- Up to 25 users
- 100GB data storage
- Unlimited dashboards
- All data source integrations (50+ connectors)
- Priority email and chat support with 4-hour response time
- Data retention: 1 year

--- 297 chars ---
- Data retention: 1 year
- API access: 10,000 calls/month
- Custom branding: Included
- Advanced analytics: Predictive modeling and anomaly detection
- Scheduled reports: Up to 50 per month
### Enterprise Plan - Custom Pricing (starting at $500/month)
- Unlimited users
- 10TB or more data storage



### The markdown splitter's default loses every price

Splitting on headers keeps each plan whole and records its product and plan in metadata. But it removes the headers from the text, and this document keeps its prices in the headers. Only the text is embedded and passed to the model, so the table above shows no chunk with a price.

In [4]:
doc = chunks_by_strategy["markdown-default"][1]
print(doc.page_content)
print("\nMetadata:", doc.metadata)

- Up to 25 users
- 100GB data storage
- Unlimited dashboards
- All data source integrations (50+ connectors)
- Priority email and chat support with 4-hour response time
- Data retention: 1 year
- API access: 10,000 calls/month
- Custom branding: Included
- Advanced analytics: Predictive modeling and anomaly detection
- Scheduled reports: Up to 50 per month

Metadata: {'Platform': 'DataFlow Platform - Pricing Guide', 'Product': 'DataFlow Analytics', 'Plan': 'Pro Plan - $99/month'}


## 2. Retrieval: one query, three chunk sizes

Before involving the model, it helps to see what each size retrieves. The Pro plan has 9 features that no other plan has, so counting how many reach the top 3 results measures how complete an answer could be.

In [5]:
QUERY = "What features are included in the Analytics Pro plan?"
PRO_FEATURES = [
    "Up to 25 users", "100GB data storage", "50+ connectors", "4-hour response time",
    "Data retention: 1 year", "10,000 calls/month", "Custom branding: Included",
    "Predictive modeling", "Up to 50 per month",
]

stores = {
    name: build_store(chunks_by_strategy[name], embeddings, name)
    for name in ("recursive-100", "recursive-300", "recursive-600")
}

for name, store in stores.items():
    results = store.similarity_search(QUERY, k=3)
    retrieved = " ".join(doc.page_content for doc in results)
    found = sum(feature in retrieved for feature in PRO_FEATURES)
    print(f"{name}: {found} of {len(PRO_FEATURES)} Pro plan features retrieved")
    for rank, doc in enumerate(results, 1):
        print(f"  {rank}. ({len(doc.page_content)} chars) {doc.page_content.splitlines()[0]}")
    print()

recursive-100: 2 of 9 Pro plan features retrieved
  1. (74 chars) Combines Analytics Pro, CRM Growth, and Storage Standard into one package.
  2. (85 chars) ### Pro Plan - $99/month
  3. (69 chars) Trial accounts have full access to all features of the selected plan.

recursive-300: 7 of 9 Pro plan features retrieved
  1. (279 chars) ### Pro Plan - $99/month
  2. (256 chars) ### Scale Plan - $99/user/month
  3. (273 chars) ## DataFlow Analytics

recursive-600: 9 of 9 Pro plan features retrieved
  1. (383 chars) ### Pro Plan - $99/month
  2. (563 chars) ### Enterprise Plan - Custom Pricing (starting at $500/month)
  3. (511 chars) ### Growth Plan - $49/user/month



## 3. A baseline agent

The natural first choice is a mid-sized recursive split with the top 4 chunks. The questions cover a plan lookup, a recommendation, a policy, a question that spans plans, an ambiguous plan name, something the document doesn't cover, and a claim that can be checked.

In [6]:
baseline = build_rag_chain(stores["recursive-300"], llm, k=4)

for question in SAMPLE_QUESTIONS:
    print(f"Q: {question}")
    print(f"A: {ask(baseline, question)}\n")

Q: What is included in the Analytics Pro plan?


A: I don't have that information.

Q: Which plan should I choose for a team of 10 that needs CRM?


A: For a team of 10 users that needs a CRM, the **Growth Plan** at **$49 per user per month** is the appropriate choice.

Q: What is the annual discount and how does it apply?


A: The annual plans offer a **20% discount** compared to the monthly billing option. This discount applies to every plan (Free Trials, Pro, and Enterprise) when you choose annual billing over monthly billing.

Q: Which plans include HIPAA compliance?


A: HIPAA compliance is available on the **Enterprise plans** only:

- **Enterprise Plan – $199/user/month**  
- **Enterprise Plan – Custom Pricing (starting at $500/month)**

Q: How much does the Enterprise plan cost?


A: The Enterprise plans are priced as follows:

- **Enterprise Plan – $199 per user per month**  
  (Includes everything in the Scale plan plus a dedicated customer success manager, custom AI models, advanced security, custom SLA, dedicated infrastructure, and unlimited API access.)

- **Enterprise Plan – Custom Pricing (starting at $500 per month)**  
  (Unlimited users, 10 TB or more data storage, unlimited dashboards, all integrations plus custom connectors, 24/7 dedicated support with SLA guarantee, 7‑year data retention, and unlimited API access.)

Q: Is there a student discount?


A: I don't have that information.

Q: How much do I save with the DataFlow Suite Bundle?


A: You save **35%** by choosing the DataFlow Suite Bundle compared to buying each product separately.



### Ground truth for the bundle

The agent must not do arithmetic here, but we can. The bundle combines Analytics Pro ($99), CRM Growth ($49 per user) and Storage Standard ($29) for $149, and the document says this saves 35%.

In [7]:
bundle_price = 149

for seats in (1, 2, 3):
    separate = 99 + 49 * seats + 29
    print(f"{seats} CRM seat(s): ${separate} bought separately, bundle saves {1 - bundle_price / separate:.1%}")

1 CRM seat(s): $177 bought separately, bundle saves 15.8%
2 CRM seat(s): $226 bought separately, bundle saves 34.1%
3 CRM seat(s): $275 bought separately, bundle saves 45.8%


### Checking the answers

Each answer, checked against the guide:

| Question | Result | Notes |
| --- | --- | --- |
| Analytics Pro plan | **Failed** | Refused, even though the Pro plan chunk was retrieved |
| Team of 10 needing CRM | Correct | Growth Plan at $49 per user per month |
| Annual discount | Partly right | 20% is correct, but it lists "Free Trials" as a plan, which the guide doesn't say |
| HIPAA | Correct | Both Enterprise plans, but neither is tied to a product |
| Enterprise cost | Partly right | Both prices, but with no product names a reader can't tell which is which |
| Student discount | Correct | Refused, because the guide doesn't cover it |
| Suite Bundle saving | Misleading | Repeats "35%". The cell above shows 15.8% with one CRM seat; about 35% needs two |

The first failure matters most, and it repeats every run. The retrieved chunk says "Pro Plan" but never "Analytics", because that heading landed in a different chunk. Told to use only the context, the model can't link them, and refusing is better than guessing. So the fault is in the chunking, not the model.

## 4. Putting the context back

The Pro plan chunk never says "Analytics", because that heading landed in a different chunk. The markdown split already keeps plans whole, so the fix is to write the product and plan back into each chunk's text. That restores the prices, and it means the words a customer uses ("Analytics Pro") are embedded too.

In [8]:
print(chunks_by_strategy["markdown-contextual"][1].page_content)
print()

contextual_store = build_store(chunks_by_strategy["markdown-contextual"], embeddings, "markdown-contextual")
contextual = build_rag_chain(contextual_store, llm, k=4)

for question in SAMPLE_QUESTIONS:
    print(f"Q: {question}")
    print(f"A: {ask(contextual, question)}\n")

DataFlow Analytics - Pro Plan - $99/month
- Up to 25 users
- 100GB data storage
- Unlimited dashboards
- All data source integrations (50+ connectors)
- Priority email and chat support with 4-hour response time
- Data retention: 1 year
- API access: 10,000 calls/month
- Custom branding: Included
- Advanced analytics: Predictive modeling and anomaly detection
- Scheduled reports: Up to 50 per month



Q: What is included in the Analytics Pro plan?


A: **DataFlow Analytics – Pro Plan ($99/month)**  

- Up to 25 users  
- 100 GB data storage  
- Unlimited dashboards  
- All data source integrations (50+ connectors)  
- Priority email and chat support with 4‑hour response time  
- Data retention: 1 year  
- API access: 10,000 calls/month  
- Custom branding: Included  
- Advanced analytics: Predictive modeling and anomaly detection  
- Scheduled reports: Up to 50 per month

Q: Which plan should I choose for a team of 10 that needs CRM?


A: For a team of 10 users, the **DataFlow CRM – Growth Plan** is the appropriate choice. It supports unlimited users, offers 50,000 contacts, advanced pipeline management, and includes email automation, standard reports, and 20+ app integrations. The cost is **$49 per user per month**.

Q: What is the annual discount and how does it apply?


A: The annual plans offer a **20% discount** compared to the monthly billing rate. This discount applies to every plan, including the Enterprise plan.

Q: Which plans include HIPAA compliance?


A: HIPAA compliance is available on all **Enterprise plans**:

- **DataFlow CRM – Enterprise Plan** ($199/user/month)  
- **DataFlow Analytics – Enterprise Plan** (Custom Pricing, starting at $500/month)

Q: How much does the Enterprise plan cost?


A: The Enterprise plan costs **$199 per user per month**.

Q: Is there a student discount?


A: I don't have that information.

Q: How much do I save with the DataFlow Suite Bundle?


A: You save **35%** by choosing the DataFlow Suite Bundle.



The Pro plan question now gets all 10 features, and the HIPAA answer names each product. But the Enterprise answer got worse: it now gives only the CRM price, where the baseline at least gave both. The next cell shows why.

In [9]:
question = "How much does the Enterprise plan cost?"

for rank, doc in enumerate(contextual_store.similarity_search(question, k=6), 1):
    print(f"{rank}. {doc.page_content.splitlines()[0]}")

1. Billing and Policies - Free Trials
2. DataFlow CRM - Enterprise Plan - $199/user/month
3. DataFlow CRM - Growth Plan - $49/user/month
4. DataFlow Analytics - Starter Plan - $29/month
5. DataFlow Analytics - Enterprise Plan - Custom Pricing (starting at $500/month)
6. DataFlow Analytics - Pro Plan - $99/month


The Analytics Enterprise plan ranks fifth, just outside the top 4, so the model never saw it. This is a retrieval miss, so the fix belongs in retrieval: raising `k` to 5 brings the plan into the context. That is the agent's default setting, shown on all seven questions below.

In [10]:
agent = build_rag_chain(contextual_store, llm, k=5)

for question in SAMPLE_QUESTIONS:
    print(f"Q: {question}")
    print(f"A: {ask(agent, question)}\n")

Q: What is included in the Analytics Pro plan?


A: **DataFlow Analytics – Pro Plan ($99/month)**  

- Up to 25 users  
- 100 GB data storage  
- Unlimited dashboards  
- All data source integrations (50+ connectors)  
- Priority email and chat support with 4‑hour response time  
- Data retention: 1 year  
- API access: 10,000 calls/month  
- Custom branding: Included  
- Advanced analytics: Predictive modeling and anomaly detection  
- Scheduled reports: Up to 50 per month

Q: Which plan should I choose for a team of 10 that needs CRM?


A: For a team of 10 users that needs CRM, you should choose the **DataFlow CRM - Growth Plan** at **$49 per user per month**.

Q: What is the annual discount and how does it apply?


A: Annual plans offer a **20% discount** compared to monthly billing. This discount applies to any plan that is billed annually.

Q: Which plans include HIPAA compliance?


A: HIPAA compliance is available on **Enterprise plans only**.  
The Enterprise plans listed in the context are:

- **DataFlow CRM – Enterprise Plan** – $199/user/month  
- **DataFlow Analytics – Enterprise Plan** – Custom Pricing (starting at $500/month)

Q: How much does the Enterprise plan cost?


A: The Enterprise plans are priced as follows:

- **DataFlow CRM – Enterprise Plan**: **$199 per user per month**  
- **DataFlow Analytics – Enterprise Plan**: **Custom pricing, starting at $500 per month**

Q: Is there a student discount?


A: I don't have that information.

Q: How much do I save with the DataFlow Suite Bundle?


A: You save **35%** by choosing the DataFlow Suite Bundle.



### One prompt line that earned its place

The prompt used throughout this notebook says that a rule stated for a group of plans applies to every plan in that group. That line was added after a failure: the guide states "HIPAA compliance is available on Enterprise plans only", the CRM Enterprise plan also mentions HIPAA itself, and the Analytics Enterprise plan doesn't. Without the line, the answer depended on an unrelated chunk.

This cell runs the same HIPAA question four times with and without the line, at both values of `k`. The context is identical in each pair. At `k=5` the only change from `k=4` is one extra chunk about free trials.

In [11]:
from langchain_core.prompts import ChatPromptTemplate

GROUP_RULE = '- A rule stated for a group of plans, such as "Enterprise plans", applies to every plan in that group.\n'
system_text = PRICING_PROMPT.messages[0].prompt.template
assert GROUP_RULE in system_text
prompt_without_rule = ChatPromptTemplate.from_messages([
    ("system", system_text.replace(GROUP_RULE, "")),
    ("human", "{question}"),
])

hipaa_question = "Which plans include HIPAA compliance?"
RUNS = 4


def names_both_enterprise_plans(answer):
    return "Analytics" in answer and "CRM" in answer


print(f"Answers naming both Enterprise plans, out of {RUNS}:\n")
print(f"{'':<16}{'k=4':>6}{'k=5':>6}")
for label, prompt in (("Without rule", prompt_without_rule), ("With rule", PRICING_PROMPT)):
    counts = []
    for k in (4, 5):
        chain = build_rag_chain(contextual_store, llm, k=k, prompt=prompt)
        counts.append(sum(names_both_enterprise_plans(ask(chain, hipaa_question)) for _ in range(RUNS)))
    print(f"{label:<16}{counts[0]:>6}{counts[1]:>6}")

Answers naming both Enterprise plans, out of 4:

                   k=4   k=5


Without rule         3     0


With rule            4     4


## 5. Scoring every strategy

Seven answers are a demo, not a measurement, so this scores every strategy on 13 questions covering 18 facts. Each fact is an exact string the context must contain, such as `$79/month` or `30-day notice`. No model is involved, so the result is the same every run.

- **Facts found:** the fact is somewhere in the retrieved chunks.
- **Attributed:** the fact is in a chunk that also names the right product. A price with no product attached is what the agent could not use in section 3.
- **Chars sent to LLM:** average context per question, which is what you pay for in tokens and in distraction.

In [12]:
compare_strategies(guide, embeddings, k=4)
print()
compare_strategies(guide, embeddings, k=5)

13 questions, 18 facts, top 4 chunks

Strategy               Chunks  Facts found  Attributed  Chars sent to LLM


fixed-300                  17           16          10               1116


recursive-100              64           12           8                283


recursive-300              22           15          10                974


recursive-600              10           17           9               1812


...

recursive-300              22           16          10               1239


recursive-600              10           17           9               2275


markdown-default           17           10           5               1202


markdown-contextual        17           18          18               1420


semantic                    8           18          18               3579


- **`markdown-contextual` is the only compact strategy with a full score.** It finds all 18 facts, each in a chunk naming its product, in about 1,100 characters per question at `k=4`.
- **`semantic` ties on score but not on cost.** One chunk holds two-thirds of the document (section 6), so it sends the model about 2.5 to 3 times as much text.
- **`recursive-600` finds 17 facts but attributes only 9.** Larger chunks keep plans together, but the plan is still separated from its product name.
- **`markdown-default` is last, and the two markdown strategies have identical boundaries.** Writing the header text back is the only difference, and it moves the score from 10 to 18.
- **Raising `k` barely moves these numbers.** The change to 5 was for the ambiguous Enterprise question, which this fact set doesn't test.

## 6. Why semantic chunking doesn't suit a price list

`SemanticChunker` compares neighbouring sentences, and it finds sentences by looking for `.`, `?` or `!`. Bullet lines have none, so most of the price list arrives as one long "sentence". Treating each line as a sentence is the obvious fix, and the second half of the cell tries it.

In [13]:
semantic_chunks = chunks_by_strategy["semantic"]
print("Default sentence splitting:")
print(f"  Chunk lengths: {[len(chunk.page_content) for chunk in semantic_chunks]}")
print(f"  Largest chunk: {max(len(c.page_content) for c in semantic_chunks) / len(guide):.0%} of the document\n")

line_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=70,
    sentence_split_regex=r"\n+",
)
line_chunks = line_chunker.create_documents([guide.strip()])

print("Each line as a sentence:")
print(f"  {len(line_chunks)} chunks, for example:")
for chunk in line_chunks[3:6]:
    print(f"    {chunk.page_content}")

Default sentence splitting:
  Chunk lengths: [2996, 186, 182, 170, 573, 81, 69, 255]
  Largest chunk: 66% of the document



Each line as a sentence:
  38 chunks, for example:
    - API access: Not included
    - Custom branding: Not included ### Pro Plan - $99/month
    - Up to 25 users


## 7. Keyword search and HyDE

Vague questions are hard in a different way: the customer's words aren't in the document at all. HyDE asks the model to write a hypothetical answer first, then searches with the question and that answer together. Its details may be invented, which is fine because the passage is used only for searching and never shown to the customer.

This uses BM25 keyword search over chunks of 250 characters, and prints the words each result matched.

In [14]:
index = build_keyword_index(split_recursive(guide, 250))
vague_question = "affordable option for a small startup with basic needs"

print("--- Keyword search alone ---\n")
print_hits(keyword_search(index, vague_question))

passage = hypothetical_answer(vague_question, llm)
print(f"--- Hypothetical answer ---\n\n{passage}\n")

print("--- Keyword search with the hypothetical answer ---\n")
print_hits(keyword_search(index, f"{vague_question} {passage}"))

--- Keyword search alone ---

[1] score 3.65, matched ['a', 'with']
### Free Trials
All paid plans include a 14-day free trial with no credit card required.
Enterprise plans include a 30-day proof-of-concept period with dedicated support.
Trial accounts have full access to all features of the selected plan.

[2] score 3.17, matched ['a', 'for']
### Payment Options
Monthly and annual billing are both available.
Annual plans offer a 20% discount compared to monthly billing.
Accepted payment methods include major credit cards, PayPal, and wire transfers for Enterprise plans.

[3] score 3.13, matched ['option']
- API access: Unlimited
- Custom branding: Full white-label
- SSO and SAML authentication
- On-premise deployment option
- Dedicated account manager



--- Hypothetical answer ---

Starter Plan – $19/month  
Up to 5 users, 10 GB of cloud storage, and email support during business hours.  
Includes core project dashboards, basic task tracking, and unlimited public integrations.  
Ideal for small teams that need reliable, budget‑friendly software without extra frills.

--- Keyword search with the hypothetical answer ---

[1] score 12.87, matched ['and', 'business', 'during', 'email', 'support']
- Chat and email support during business hours

[2] score 12.70, matched ['5', 'and', 'basic', 'email', 'plan', 'starter', 'support', 'to', 'up', 'with']
### Starter Plan - $29/month
- Up to 5 users
- 10GB data storage
- Basic dashboards (up to 10)
- CSV and Excel data import
- Email support with 48-hour response time
- Data retention: 90 days
- API access: Not included

[3] score 11.79, matched ['and', 'for', 'need', 'teams', 'that', 'to']
### DataFlow Suite Bundle - $149/month
Combines Analytics Pro, CRM Growth, and Storage Standard into one pa

## Observations

The chunk statistics and the scoring table are the same on every run. Answers and the hypothetical passage come from the model, so they can vary a little.

- **Chunking decided whether the agent could answer at all.** The baseline refused the most basic question. With the same model and the same kind of prompt, the final agent gets 6 of the 7 sample questions right. The seventh repeats the guide's "35%" claim.
- **Cut along the document's own structure.** The headers already mark the units a customer asks about. Sizes only tune how often those units get broken.
- **A library default can hide a serious failure.** `MarkdownHeaderTextSplitter` looks ideal until you notice that no chunk contains a price. Nothing warns you, and `strip_headers=False` isn't enough because the product header appears only once per product.
- **The same symptom can have different causes.** A plan was missing from two answers. Enterprise was a retrieval miss, fixed by raising `k`. HIPAA was the model's reading: both plans were in the context, but without the group rule it named both only 3 of 4 times at `k=4` and at most 1 of 4 at `k=5`, depending on the run. With the rule it was 4 of 4 on every run. Looking at the retrieved context is how you tell them apart.
- **Semantic chunking suits prose, not price lists.** With bullets as input it produced one chunk holding 66% of the document, and splitting by line gave fragments that glue a feature to the next plan's heading.
- **HyDE helps vague questions, but not cleanly.** Keyword search alone matched filler words (`a`, `with`, `for`) and found no entry-level plan. With the hypothetical answer, the Analytics Starter plan reached the top 3, but a short CRM line ranked above it and the result changes with the generated text.
- **RAG repeats what the source says.** Both agents quote "35%" faithfully, and neither can notice that the list prices don't support it for one seat.

## Limitations and next steps

- **Small evaluation.** 13 hand-written questions on one 4.5 KB document show a direction, not a benchmark. Attributed also rewards exactly what `markdown-contextual` adds.
- **It relies on clean headers.** This strategy works because the document has them. A messy PDF would need a different approach.
- **Rank order is fragile.** The Enterprise failure was one rank away, and raising `k` is a blunt fix. Using the `Plan` metadata to fetch every plan with a matching name, or reranking, would be sturdier.
- **No arithmetic.** The agent repeats whatever the document claims. Checking a saving needs a calculation tool, which is the next stage's job.
- **Not connected to stage 1.** This agent answers questions about a price list; it doesn't yet feed the selling-price recommender.